# Notebook 00: Your First AI Call

Welcome! This notebook gets your environment ready and makes your first LLM call. No agentic concepts yet — just setup and confidence-building.

**Expected time:** ~20 minutes
**Prerequisites:** Python 3.10+, pip


## What You'll Learn

1. Confirm your Python environment works
2. Install required packages
3. Choose and configure an LLM provider (OpenAI, DeepSeek, Groq, or OpenRouter)
4. Make your first LLM call
5. Compare how different prompts change the output
6. Handle common errors


## Setup: Install Dependencies

Run the cell below to install everything needed for this notebook.

In [1]:
!pip install openai==1.68.2 groq==0.18.0


## Step 1: Environment Check

Run this to verify Python and key packages are ready.

In [2]:
import sys
import importlib

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

required = ["openai", "groq"]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
        print(f"  {pkg}: {ver}  OK")
    except ImportError:
        print(f"  {pkg}: NOT FOUND — run the install cell above")

print("\nEnvironment check complete.")


Python version: 3.11.0 | packaged by conda-forge | (main, Jan 14 2023, 12:27:40) [GCC 11.3.0]
Python executable: /home/filpa/.local/share/mamba/envs/best_env/bin/python3.11
  openai: 1.68.2  OK
  groq: 0.18.0  OK

Environment check complete.


## Step 2: Choose Your Provider

This notebook supports **4 LLM providers**. Pick one below:

| Provider | SDK | Model | Get API Key |
|----------|-----|-------|------------|
| **OpenAI** | `openai` | `gpt-4o-mini` | https://platform.openai.com/api-keys |
| **DeepSeek** | `openai` (compatible) | `deepseek-chat` | https://platform.deepseek.com/api_keys |
| **Groq** | `groq` | `llama-3.3-70b-versatile` | https://console.groq.com/keys |
| **OpenRouter** | `openai` (compatible) | `nvidia/nemotron-3-ultra-550b-a55b:free` | https://openrouter.ai/keys |

Each provider has its own **mock flag** below. Set yours to `True` if you have no API key for that provider.

> **Using .env?** Copy `.env.example` to `.env` and fill in your keys. The notebook auto-loads keys from `.env` so you never paste them in code.
> **No API keys at all?** Leave all `MOCK` flags `True`. The mock LLM returns a canned response so you can follow along.

In [3]:
import os
import getpass
from pathlib import Path
from openai import OpenAI
from groq import Groq

# --- Optional .env file loading (no extra pip installs needed) ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")
    print("  Tip: cp .env.example .env  and fill in your keys to skip prompts.")

# --- Pick your provider ---
PROVIDER = "openrouter"  # Choose: openai, deepseek, groq, openrouter
# --------------------------

# Per-provider mock flags (True = use canned response, no API key needed)
MOCK = {
    "openai":     True,
    "deepseek":   True,
    "groq":       True,
    "openrouter": False,
}

# Environment variable names for auto-loading keys (set in .env)
KEY_ENV_VAR = {
    "openai":     "OPENAI_API_KEY",
    "deepseek":   "DEEPSEEK_API_KEY",
    "groq":       "GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

# Provider configuration table
CONFIG = {
    "openai": {
        "name": "OpenAI",
        "pkg": "openai",
        "model": "gpt-4o-mini",
        "key_url": "https://platform.openai.com/api-keys",
        "client_fn": lambda k: OpenAI(api_key=k),
        "mock_response": "Silicon dreams stir\n Circuits learn in silent hum\n Minds of code awake.",
    },
    "deepseek": {
        "name": "DeepSeek",
        "pkg": "openai",
        "model": "deepseek-chat",
        "key_url": "https://platform.deepseek.com/api_keys",
        "client_fn": lambda k: OpenAI(api_key=k, base_url="https://api.deepseek.com"),
        "mock_response": "Deep thought stirs\n  Patterns emerge in silicon sea\n  Mind awakens.",
    },
    "groq": {
        "name": "Groq",
        "pkg": "groq",
        "model": "llama-3.3-70b-versatile",
        "key_url": "https://console.groq.com/keys",
        "client_fn": lambda k: Groq(api_key=k),
        "mock_response": "Fast inference\n  LLM responds at lightning speed\n  Groq hardware hums.",
    },
    "openrouter": {
        "name": "OpenRouter",
        "pkg": "openai",
        "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
        "key_url": "https://openrouter.ai/keys",
        "client_fn": lambda k: OpenAI(api_key=k, base_url="https://openrouter.ai/api/v1"),
        "mock_response": "Nemotron thinks deep\n  Ultra-scale reasoning flows\n  Knowledge unbound.",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled. Using canned responses.")
    client = None
else:
    try:
        importlib = __import__("importlib")
        importlib.import_module(cfg["pkg"])
    except ImportError:
        print(f'Package \'{cfg["pkg"]}\' not installed. Run the install cell above.')
        print("Falling back to mock mode.")
        client = None
        use_mock = True
    else:
        api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
        client = cfg["client_fn"](api_key)
        print(f"[{cfg['name']}] API key configured.")


Loaded .env file.
[OpenRouter] API key configured.


### Validate Your Connection

This makes a tiny test call to confirm your key and provider work together.

In [4]:
if not use_mock and client is not None:
    try:
        response = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": "Say OK and nothing else."}],
            max_tokens=10,
        )
        print(f"Connection valid. Response: {response.choices[0].message.content}")
    except Exception as e:
        print(f"Validation FAILED: {type(e).__name__}: {e}")
        print(f"Check your key at {cfg['key_url']}")
else:
    print(f"[{cfg['name']}] Mock mode — skipping validation.")


Connection valid. Response: The user wants me to respond with only "OK


## Step 3: Your First LLM Call

Let's make a real call. We'll ask the model to write a haiku.

In [5]:
if not use_mock and client is not None:
    response = client.chat.completions.create(
        model=cfg["model"],
        messages=[
            {"role": "user", "content": "Write a haiku about artificial intelligence."}
        ],
    )
    reply = response.choices[0].message.content
    print("LLM response:")
    print(reply)
    print(f"\n---\nModel used: {response.model}")
    print(f"Tokens: {response.usage.total_tokens} (input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens})")
else:
    reply = cfg["mock_response"]
    print(f"[{cfg['name']}] MOCK response:")
    print(reply)


LLM response:
Silicon minds wake,  
Learning patterns, vast and deep,  
Dreams in code take flight.

---
Model used: nvidia/nemotron-3-ultra-550b-a55b-20260604:free
Tokens: 73 (input: 24, output: 49)


### Inspecting the Response

The API returns a structured object. Let's look at its parts.

In [6]:
if not use_mock and client is not None:
    response = client.chat.completions.create(
        model=cfg["model"],
        messages=[{"role": "user", "content": "Say hello world."}],
    )
    print("Full response object fields:")
    print(f"  id: {response.id}")
    print(f"  model: {response.model}")
    print(f"  created: {response.created}")
    print(f"  role: {response.choices[0].message.role}")
    print(f"  content: {response.choices[0].message.content}")
    print(f"  finish_reason: {response.choices[0].finish_reason}")
    print(f"  total_tokens: {response.usage.total_tokens}")
else:
    print(f"[{cfg['name']}] Mock mode — response object not available.")
    print("\nReal response contains: id, model, created, choices[].message.content, usage")


Full response object fields:
  id: gen-1783805622-H8ptiAhSB7mIjCwFIDuF
  model: nvidia/nemotron-3-ultra-550b-a55b-20260604:free
  created: 1783805622
  role: assistant
  content: Hello world!
  finish_reason: stop
  total_tokens: 35


## Step 4: Prompt Engineering Teaser

The same model gives different answers depending on *how* you ask. Let's compare short vs. detailed prompts.

In [7]:
if not use_mock and client is not None:
    prompts = [
        "What is agentic AI?",
        "Explain agentic AI to a first-year university student. Use an analogy. Keep it under 100 words.",
    ]
    for i, prompt in enumerate(prompts, 1):
        print(f"--- Prompt {i}: \"{prompt}\" ---")
        response = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": prompt}],
        )
        print(response.choices[0].message.content)
        print(f"(Tokens: {response.usage.total_tokens})\n")
else:
    print(f"[{cfg['name']}] Mock mode:")
    print("  Prompt 1 (short): Agentic AI is AI that can take actions beyond just generating text.")
    print("  Prompt 2 (detailed): Imagine a librarian who doesn't just answer questions but actually goes to find the book, opens it, and reads you the relevant page. That's agentic AI.")


--- Prompt 1: "What is agentic AI?" ---
**Agentic AI** refers to artificial intelligence systems that possess **agency**—the ability to act independently, make decisions, and pursue complex goals with limited or no direct human supervision.

Unlike traditional AI models (like standard chatbots) that simply *respond* to a prompt, agentic AI systems *plan*, *execute*, *observe*, and *iterate* to achieve an objective.

---

### 1. Core Definition: The Shift from "Chat" to "Act"
| Feature | **Traditional LLM (Chatbot)** | **Agentic AI** |
| :--- | :--- | :--- |
| **Role** | Passive Responder | Active Problem Solver |
| **Input** | Single prompt | High-level Goal / Objective |
| **Process** | Generate text prediction | **Plan → Act → Observe → Reflect → Repeat** |
| **Tools** | None (internal knowledge only) | **Tool Use** (Code, Search, APIs, Databases, Other Agents) |
| **Memory** | Context window only | **Short-term (context) + Long-term (vector DBs, knowledge graphs)** |
| **Autonomy** 

In [8]:
if not use_mock and client is not None:
    print("--- With vs. Without System Prompt ---")
    
    r1 = client.chat.completions.create(
        model=cfg["model"],
        messages=[{"role": "user", "content": "Explain gradient descent in 2 sentences."}],
    )
    print("No system prompt:")
    print(r1.choices[0].message.content)
    
    r2 = client.chat.completions.create(
        model=cfg["model"],
        messages=[
            {"role": "system", "content": "You are a physics professor. Use precise mathematical language."},
            {"role": "user", "content": "Explain gradient descent in 2 sentences."},
        ],
    )
    print("\nWith system prompt (physics professor):")
    print(r2.choices[0].message.content)
else:
    print(f"[{cfg['name']}] Mock mode:")
    print("  Without: Gradient descent iteratively adjusts parameters to minimize a loss function.")
    print("  With: Let us consider the gradient operator ∇ acting on the loss surface L(θ)...")


--- With vs. Without System Prompt ---
No system prompt:
Gradient descent is an iterative optimization algorithm that minimizes a loss function by repeatedly adjusting parameters in the direction of the steepest descent, calculated as the negative gradient. At each step, the parameters are updated by subtracting the gradient scaled by a learning rate, gradually converging toward a local or global minimum.

With system prompt (physics professor):
Gradient descent is an iterative optimization algorithm that minimizes a differentiable objective function $f(\mathbf{x})$ by updating the parameters in the direction opposite to the gradient, $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta \nabla f(\mathbf{x}_k)$, where $\eta > 0$ is the learning rate. Under suitable convexity and Lipschitz continuity conditions on $\nabla f$, this first-order method guarantees convergence to a stationary point where the gradient vanishes.


## Troubleshooting: Common Errors

Let's break things on purpose so you recognize the errors when they happen.

### Error 1: Missing or Invalid API Key

What happens when you pass a bad key?

In [9]:
bad_client = OpenAI(api_key="sk-bad-key")
try:
    bad_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "hello"}],
    )
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")
    print("\nFix: Generate a valid key at the provider's API key page.")


Error type: AuthenticationError
Error message: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-bad-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

Fix: Generate a valid key at the provider's API key page.


### Error 2: Wrong Model Name

Model names are exact strings. A typo = an error.

In [10]:
if not use_mock and client is not None:
    try:
        client.chat.completions.create(
            model="gpt-4o-mini",  # intentional typo: should match cfg["model"]
            messages=[{"role": "user", "content": "hello"}],
        )
    except Exception as e:
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {e}")
        print("\nFix: Check the model name in cfg['model'] against provider docs.")
else:
    print(f"[{cfg['name']}] Mock mode — skipping error demo.")
    print("In real mode you'd see an error like: NotFoundError or BadRequestError")


### Error 3: Empty Messages

The API requires at least one message.

In [11]:
if not use_mock and client is not None:
    try:
        client.chat.completions.create(
            model=cfg["model"],
            messages=[],
        )
    except Exception as e:
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {e}")
        print("\nFix: Always include at least one message in the messages list.")
else:
    print(f"[{cfg['name']}] Mock mode — skipping error demo.")


Error type: BadRequestError
Error message: Error code: 400 - {'error': {'message': 'Input required: specify "prompt" or "messages"', 'code': 400}, 'user_id': 'user_3GNE1txMIZj2s67bcrZAGElnyfb'}

Fix: Always include at least one message in the messages list.


## Exercises

Choose your tier. Bronze is run-only (no coding). Silver modifies existing code. Gold builds something new.


### Bronze — Run and Observe

Run the cell below. It makes 3 different prompts automatically using your chosen provider. Observe how the output changes.

In [12]:
if not use_mock and client is not None:
    topics = [
        "machine learning",
        "database systems",
        "cybersecurity",
    ]
    for topic in topics:
        response = client.chat.completions.create(
            model=cfg["model"],
            messages=[{
                "role": "user",
                "content": f"Define {topic} in one sentence for a beginner."
            }],
        )
        print(f"{topic}: {response.choices[0].message.content}\n")
else:
    print(f"[{cfg['name']}] Mock mode responses:")
    print("  machine learning: Teaching computers to learn from data without explicit instructions.")
    print("  database systems: Organized collections of data with efficient storage and retrieval.")
    print("  cybersecurity: Protecting systems, networks, and data from digital attacks.")


machine learning: Machine learning is teaching computers to learn patterns from data so they can make predictions or decisions without being explicitly programmed for every specific rule.

database systems: A database system is an organized digital filing cabinet managed by software that lets you easily store, find, update, and secure large amounts of related information.

cybersecurity: Cybersecurity is the practice of protecting computers, networks, and data from digital attacks, theft, or unauthorized access.



### Silver — Change the Model and Provider

Edit the `PROVIDER`, `model` (in `CONFIG`), and prompt variables below. Try a different provider or model. Try a prompt from your own domain.

In [13]:
# --- Edit these lines ---
silver_provider = "openai"      # Try: deepseek, groq, openrouter
silver_model = "gpt-4o-mini"    # Must match the provider's available models
silver_prompt = "Explain why agentic AI matters for biomedical research."
# --------------------------

silver_cfg = CONFIG.get(silver_provider)
if silver_cfg is None:
    print(f"Unknown provider: {silver_provider}")
elif MOCK.get(silver_provider, True):
    print(f"[{silver_cfg['name']}] Mock mode — skipping real call.")
    print(f"Model: {silver_model}")
    print(f"Prompt: {silver_prompt}")
    print("[Mock] In real mode, you'd see the actual response here.")
else:
    try:
        importlib.import_module(silver_cfg["pkg"])
        silver_key = getpass.getpass(f"Enter {silver_cfg['name']} API key: ")
        silver_client = silver_cfg["client_fn"](silver_key)
        print(f"Using: {silver_cfg['name']} / {silver_model}")
        print(f"Prompt: {silver_prompt}\n")
        response = silver_client.chat.completions.create(
            model=silver_model,
            messages=[{"role": "user", "content": silver_prompt}],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")


[OpenAI] Mock mode — skipping real call.
Model: gpt-4o-mini
Prompt: Explain why agentic AI matters for biomedical research.
[Mock] In real mode, you'd see the actual response here.


### Gold — Compare All Four Providers

Create clients for all 4 providers (mock or real). Ask the same question to each. Compare verbosity, tone, and content across providers.

In [14]:
gold_question = "What are the ethical risks of autonomous AI agents?"

for prov_id, prov_cfg in CONFIG.items():
    name = prov_cfg["name"]
    print(f"--- {name} ---")
    if MOCK.get(prov_id, True):
        print(f"[Mock] Would respond to: {gold_question[:50]}...")
        print()
        continue
    try:
        importlib.import_module(prov_cfg["pkg"])
        pk = getpass.getpass(f"Enter {name} API key: ")
        prov_client = prov_cfg["client_fn"](pk)
        resp = prov_client.chat.completions.create(
            model=prov_cfg["model"],
            messages=[{"role": "user", "content": gold_question}],
        )
        print(resp.choices[0].message.content)
        print(f"(Tokens: {resp.usage.total_tokens})\n")
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}\n")


--- OpenAI ---
[Mock] Would respond to: What are the ethical risks of autonomous AI agents...

--- DeepSeek ---
[Mock] Would respond to: What are the ethical risks of autonomous AI agents...

--- Groq ---
[Mock] Would respond to: What are the ethical risks of autonomous AI agents...

--- OpenRouter ---


Enter OpenRouter API key:  ········


The ethical risks of autonomous AI agents—systems capable of perceiving environments, making decisions, and taking actions to achieve goals with minimal human oversight—are profound because they shift AI from a **tool** (passive, human-directed) to an **actor** (active, self-directed).

Here is a structured breakdown of the primary ethical risks:

---

### 1. Accountability & Responsibility Gaps ("The Liability Vacuum")
*   **Moral Crumple Zones:** When an autonomous agent causes harm (e.g., a trading bot crashes a market, a self-driving car hits a pedestrian), it is unclear who is liable: the developer, the deployer, the user, or the entity that trained the model.
*   **Diffusion of Responsibility:** Complex supply chains (data providers → model trainers → fine-tuners → deployers → users) dilute accountability. No single party has full visibility or control over the agent’s final behavior.
*   **Legal Personhood Debate:** Current legal frameworks assume human actors. Granting agents l

## Next Steps

You've confirmed your environment works and made your first LLM calls. You've seen how prompts change output and tried multiple providers.

**Next notebook:** `01_agent_loop.ipynb` — "The Agent Loop"

> "Now that you can call an LLM, let's make it loop."
